# Libraries

In [2]:
import joblib
import torch
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [3]:
cleveland = pd.read_csv("data/processed/cleveland_clean.csv")
hungarian = pd.read_csv("data/processed/hungarian_clean.csv")
switzerland = pd.read_csv("data/processed/switzerland_clean.csv")
va = pd.read_csv("data/processed/va_clean.csv")

df = pd.concat(
    [cleveland, hungarian, switzerland, va],
    ignore_index=True
)

df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


In [3]:
X = df.drop("target", axis=1)
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [4]:
xgb_model = joblib.load(
    "models/xgboost_best.pkl"
)

FileNotFoundError: [Errno 2] No such file or directory: 'models/xgboost_best.pkl'

In [ ]:
y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)[:,1]

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_prob)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("ROC AUC  :", roc)

In [ ]:
print(classification_report(y_test,y_pred))

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred
)

plt.figure(figsize=(6,5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("XGBoost Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.savefig("results/confusion_matrix.png")
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(
    y_test,
    y_prob
)

plt.figure(figsize=(6,5))
plt.plot(
    fpr,
    tpr,
    label=f"AUC = {roc:.3f}"
)

plt.plot(
    [0,1],
    [0,1],
    "--"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.savefig(
    "results/roc_curve.png"
)
plt.show()

In [ ]:
class HeartDiseaseNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.network = torch.nn.Sequential(
            torch.nn.Linear(13,32),
            torch.nn.ReLU(),
            torch.nn.Linear(32,16),
            torch.nn.ReLU(),
            torch.nn.Linear(16,1),
            torch.nn.Sigmoid()
        )

    def forward(self,x):
        return self.network(x)

In [ ]:
fl_model = HeartDiseaseNet()
fl_model.load_state_dict(
    torch.load(
        "models/global_model.pth",
        map_location="cpu"
    )
)
fl_model.eval()

In [ ]:
X_tensor = torch.tensor(
    X_test,
    dtype=torch.float32
)

with torch.no_grad():

    pred = fl_model(
        X_tensor
    ).numpy()

pred_class = (
    pred>0.5
).astype(int)

fl_accuracy = accuracy_score(
    y_test,
    pred_class
)

print(fl_accuracy)

In [ ]:
comparison = pd.DataFrame({

    "Model":[
        "Logistic Regression",
        "Random Forest",
        "XGBoost",
        "Federated Learning",
        "Federated + DP"
    ],

    "Accuracy":[
        0.815,
        0.842,
        0.837,
        fl_accuracy,
        dp_accuracy    # Replace with actual value
    ]

})

comparison

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(

    data=comparison,

    x="Model",

    y="Accuracy"

)

plt.xticks(rotation=20)

plt.tight_layout()

plt.savefig(
    "results/model_comparison.png"
)

plt.show()

In [ ]:
import shap

explainer = shap.TreeExplainer(xgb_model)

shap_values = explainer.shap_values(X_test)

shap.summary_plot(
    shap_values,
    X_test,
    feature_names=X.columns
)

In [ ]:
plt.savefig(
    "results/shap_summary.png",
    bbox_inches="tight"
)